In [5]:
from pathlib import Path
import pandas as pd
import folium
from folium import FeatureGroup
from folium.plugins import PolyLineTextPath
from IPython.display import display, IFrame


# =========================================================
# helper
# =========================================================

def add_solution_to_map(
    m,
    result_dir,
    solution_name,
    color_palette
):

    result_dir = Path(result_dir)

    deliveries = pd.read_csv(result_dir / "deliveries.csv")
    routes = pd.read_csv(result_dir / "routes.csv")
    cafes = pd.read_csv(result_dir / "served_cafes.csv")

    summary = (
        deliveries
        .groupby(["van_id", "cafe_name"], as_index=False)
        .agg(
            total_weight_kg=("total_weight_kg", "sum"),
            total_margin=("total_margin", "sum"),
            n_products=("product_id", "nunique")
        )
    )

    cafes_plot = cafes.merge(
        summary,
        left_on=["van_id", "name"],
        right_on=["van_id", "cafe_name"],
        how="left"
    )

    max_margin = cafes_plot["total_margin"].fillna(0).max()

    solution_group = FeatureGroup(
        name=f"{solution_name}",
        show=True
    )

    # -------------------------
    # depot
    # -------------------------
    depot_rows = cafes_plot[cafes_plot["node_type"] == "depot"]

    for _, row in depot_rows.iterrows():

        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            popup=f"{solution_name} Depot",
            tooltip=f"{solution_name} Depot",
            icon=folium.Icon(
                color="black",
                icon="home",
                prefix="fa"
            )
        ).add_to(solution_group)

    # -------------------------
    # routes
    # -------------------------
    for van_id in sorted(routes["van_id"].dropna().unique()):

        color = color_palette.get(van_id, "gray")

        van_routes = routes[routes["van_id"] == van_id]

        for _, r in van_routes.iterrows():

            locations = [
                [r["from_latitude"], r["from_longitude"]],
                [r["to_latitude"], r["to_longitude"]],
            ]

            line = folium.PolyLine(
                locations=locations,
                color=color,
                weight=4,
                opacity=0.75,
                tooltip=(
                    f"{solution_name}<br>"
                    f"{van_id}<br>"
                    f"{r['from_node']} → {r['to_node']}<br>"
                    f"{r['distance_km']:.2f} km"
                )
            )

            line.add_to(solution_group)

            PolyLineTextPath(
                line,
                " → ",
                repeat=True,
                offset=7,
                attributes={
                    "fill": color,
                    "font-size": "12",
                    "font-weight": "bold"
                }
            ).add_to(solution_group)

    # -------------------------
    # cafes
    # -------------------------
    cafe_rows = cafes_plot[cafes_plot["node_type"] == "cafe"]

    for _, row in cafe_rows.iterrows():

        margin = (
            0 if pd.isna(row["total_margin"])
            else row["total_margin"]
        )

        radius = 5 + 10 * (
            margin / max_margin
            if max_margin > 0 else 0
        )

        cafe_delivery = deliveries[
            (deliveries["van_id"] == row["van_id"]) &
            (deliveries["cafe_name"] == row["name"])
        ]

        product_table = cafe_delivery[
            ["product_id", "quantity", "total_margin"]
        ].to_html(
            index=False,
            float_format=lambda x: f"{x:.2f}",
            border=0
        )

        popup = f"""
        <b>{solution_name}</b><br>
        <b>{row['name']}</b><br>
        Van: {row['van_id']}<br>
        Weight: {row['total_weight_kg']:.2f} kg<br>
        Margin: ${row['total_margin']:.2f}<br>
        Products: {row['n_products']}<br><br>
        {product_table}
        """

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            color=color_palette.get(row["van_id"], "gray"),
            fill=True,
            fill_opacity=0.85,
            popup=folium.Popup(popup, max_width=450),
            tooltip=(
                f"{solution_name} | "
                f"{row['name']} | "
                f"${margin:.2f}"
            )
        ).add_to(solution_group)

    solution_group.add_to(m)


# =========================================================
# create ONE map
# =========================================================

m = folium.Map(
    location=[-37.80, 144.97],
    zoom_start=12,
    tiles="CartoDB positron"
)

folium.TileLayer(
    "OpenStreetMap",
    name="OpenStreetMap"
).add_to(m)

folium.TileLayer(
    "CartoDB dark_matter",
    name="Dark"
).add_to(m)

# MTZ colors
mtz_colors = {
    "van_1": "purple",
    "van_2": "blue",
    "van_3": "green",
    "van_4": "orange",
    "van_5": "red",
    "van_6": "darkred",
}

# DFJ colors
dfj_colors = {
    "van_1": "cadetblue",
    "van_2": "lightblue",
    "van_3": "lightgreen",
    "van_4": "beige",
    "van_5": "pink",
    "van_6": "lightgray",
}

# add MTZ
add_solution_to_map(
    m,
    "../results/mtz",
    "MTZ",
    mtz_colors
)

# add DFJ
add_solution_to_map(
    m,
    "../results/dfj",
    "DFJ",
    dfj_colors
)

# layer toggle
folium.LayerControl(
    collapsed=False
).add_to(m)

# save
OUT_PATH = Path("../results/comparison_map.html")

m.save(OUT_PATH)

print("Saved to:")
print(OUT_PATH)

display(
    IFrame(
        str(OUT_PATH),
        width=1200,
        height=800
    )
)

Saved to:
..\results\comparison_map.html


In [4]:
from pathlib import Path
import pandas as pd
import folium
from IPython.display import display, IFrame

# =========================================================
# Generic function to draw routing solution map
# =========================================================

def build_route_map(result_dir, output_name):
    """
    result_dir:
        ../results/mtz
        ../results/dfj
    """

    result_dir = Path(result_dir)

    deliveries_path = result_dir / "deliveries.csv"
    routes_path = result_dir / "routes.csv"
    cafes_path = result_dir / "served_cafes.csv"

    # -------------------------
    # read csv
    # -------------------------
    deliveries = pd.read_csv(deliveries_path)
    routes = pd.read_csv(routes_path)
    cafes = pd.read_csv(cafes_path)

    # -------------------------
    # aggregate cafe info
    # -------------------------
    summary = (
        deliveries
        .groupby(["van_id", "cafe_name"], as_index=False)
        .agg(
            total_weight_kg=("total_weight_kg", "sum"),
            total_margin=("total_margin", "sum"),
            n_products=("product_id", "nunique")
        )
    )

    cafes_plot = cafes.merge(
        summary,
        left_on=["van_id", "name"],
        right_on=["van_id", "cafe_name"],
        how="left"
    )

    # -------------------------
    # create base map
    # -------------------------
    center = [
        cafes_plot["latitude"].mean(),
        cafes_plot["longitude"].mean()
    ]

    m = folium.Map(
        location=center,
        zoom_start=12,
        tiles="CartoDB positron"
    )

    # -------------------------
    # van colors
    # -------------------------
    van_colors = {
        "van_1": "purple",
        "van_2": "blue",
        "van_3": "green",
        "van_4": "orange",
        "van_5": "red",
        "van_6": "darkred",
    }

    # -------------------------
    # draw routes
    # -------------------------
    for _, r in routes.iterrows():

        color = van_colors.get(r["van_id"], "gray")

        tooltip = (
            f"{r['van_id']}<br>"
            f"{r['from_node']} → {r['to_node']}<br>"
            f"Distance: {r['distance_km']:.2f} km<br>"
            f"Travel time: {r['travel_time_hours']:.2f} h"
        )

        folium.PolyLine(
            locations=[
                [r["from_latitude"], r["from_longitude"]],
                [r["to_latitude"], r["to_longitude"]],
            ],
            color=color,
            weight=4,
            opacity=0.75,
            tooltip=tooltip
        ).add_to(m)

    # -------------------------
    # draw cafes
    # -------------------------
    for _, row in cafes_plot.iterrows():

        if row["node_type"] == "depot":

            folium.Marker(
                location=[row["latitude"], row["longitude"]],
                popup=f"<b>{row['name']}</b><br>Depot",
                tooltip="Depot",
                icon=folium.Icon(
                    color="black",
                    icon="home",
                    prefix="fa"
                )
            ).add_to(m)

        else:

            color = van_colors.get(row["van_id"], "gray")

            popup = f"""
            <b>{row['name']}</b><br>
            Assigned van: {row['van_id']}<br>
            Total weight: {row['total_weight_kg']:.2f} kg<br>
            Total margin: ${row['total_margin']:.2f}<br>
            Product types: {row['n_products']}
            """

            folium.CircleMarker(
                location=[row["latitude"], row["longitude"]],
                radius=6,
                color=color,
                fill=True,
                fill_opacity=0.85,
                popup=folium.Popup(popup, max_width=300),
                tooltip=row["name"]
            ).add_to(m)

    # -------------------------
    # legend
    # -------------------------
    legend_html = """
    <div style="
    position: fixed;
    bottom: 40px;
    left: 40px;
    width: 180px;
    background-color: white;
    border:2px solid grey;
    z-index:9999;
    font-size:14px;
    padding: 10px;
    ">
    <b>Vehicle Routes</b><br>
    <span style="color:purple;">●</span> van_1<br>
    <span style="color:blue;">●</span> van_2<br>
    <span style="color:green;">●</span> van_3<br>
    <span style="color:orange;">●</span> van_4<br>
    <span style="color:red;">●</span> van_5<br>
    <span style="color:darkred;">●</span> van_6<br>
    <span style="color:black;">●</span> depot
    </div>
    """

    m.get_root().html.add_child(
        folium.Element(legend_html)
    )

    # -------------------------
    # save
    # -------------------------
    out_path = result_dir / output_name
    m.save(out_path)

    return m, out_path


# =========================================================
# Build MTZ map
# =========================================================

mtz_map, mtz_path = build_route_map(
    "../results/mtz",
    "mtz_route_map.html"
)

print("MTZ map saved to:")
print(mtz_path)

display(
    IFrame(
        str(mtz_path),
        width=1000,
        height=700
    )
)


# =========================================================
# Build DFJ map
# =========================================================

dfj_map, dfj_path = build_route_map(
    "../results/dfj",
    "dfj_route_map.html"
)

print("DFJ map saved to:")
print(dfj_path)

display(
    IFrame(
        str(dfj_path),
        width=1000,
        height=700
    )
)

MTZ map saved to:
..\results\mtz\mtz_route_map.html


DFJ map saved to:
..\results\dfj\dfj_route_map.html
